In [1]:
from abc import ABC, abstractmethod
from datetime import datetime, timedelta

In [2]:
class Book:

    total_books = 0        # class attribute

    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages
        self.borrowed = False
        Book.total_books += 1

    def __len__(self):
        return self.pages

    def __eq__(self, other):
        return self.title == other.title

    def __lt__(self, other):
        return self.pages < other.pages

    def __gt__(self, other):
        return self.pages > other.pages

    def __add__(self, other):
        return self.pages + other.pages

    def __str__(self):
        return f"{self.title} by {self.author} ({self.pages} pages)"
    


In [3]:
class BorrowRecord:

    def __init__(self, book, days=7):
        self.book = book
        self.borrow_date = datetime.now()
        self.due_date = self.borrow_date + timedelta(days=days)

    def overdue_days(self):
        today = datetime.now()
        if today > self.due_date:
            return (today - self.due_date).days
        return 0

    def __str__(self):
        return f"{self.book.title} due on {self.due_date.date()}"



In [4]:
class Member(ABC):

    def __init__(self, name):
        self.name = name
        self.borrowed = []

    @abstractmethod
    def fine(self, days):
        pass

    def borrow(self, book):

        if book.borrowed:
            print("Book already borrowed")
            return

        record = BorrowRecord(book)
        self.borrowed.append(record)
        book.borrowed = True

    def return_book(self, book):

        for record in self.borrowed:
            if record.book == book:

                days = record.overdue_days()
                fine = self.fine(days)

                print("Fine:", fine)

                self.borrowed.remove(record)
                book.borrowed = False
                return

    def __len__(self):
        return len(self.borrowed)

    def __getitem__(self, index):
        return self.borrowed[index]

    def __setitem__(self, index, value):
        self.borrowed[index] = value

    def __iter__(self):
        self._i = 0
        return self

    def __next__(self):
        if self._i >= len(self.borrowed):
            raise StopIteration
        val = self.borrowed[self._i]
        self._i += 1
        return val

    def __str__(self):
        return f"Member: {self.name}"


In [5]:
class BasicMember(Member):

    limit = 2

    def fine(self, days):
        return days * 5


class PremiumMember(BasicMember):

    limit = 5

    def fine(self, days):
        return days * 2


class VIPMember(PremiumMember):

    limit = 10

    def fine(self, days):
        return days * 0.5


In [6]:
class DigitalAccess:

    def download(self, book):
        print("Downloading:", book.title)


class VIPDigitalMember(VIPMember, DigitalAccess):
    pass



In [7]:
class Library:

    def search(self, *args):

        if len(args) == 1:
            print("Searching by title:", args[0])

        elif len(args) == 2:
            print("Searching by author:", args[0], args[1])

        else:
            print("Invalid search")


In [8]:
def display_item(item):
    print(item)

In [9]:
b1 = Book("AI Basics", "Andrew Ng", 300)
b2 = Book("Python Deep Dive", "Mark Lutz", 700)

m = VIPDigitalMember("Ali")

m.borrow(b1)

for record in m:
    print(record)

m.return_book(b1)

display_item(b1)     # duck typing

AI Basics due on 2026-03-15
Fine: 0.0
AI Basics by Andrew Ng (300 pages)
